# Week 4: DPO / SimPO / Steering — Solution

USSE Noosphere — Subsystem 04: Plasma Ignition & Ethical Steering Core  
Course: Conversational AI — Winter Semester 2026/27  
Waypoint: Betelgeuse & Orion Molecular Cloud Complex (SEC-04-ORI)

This notebook drives the four standalone deliverables:
1. `01_DPO_Loss_Core.py` — pure PyTorch DPO loss + gradient isolation test
2. `02_Implicit_Reward_Evaluator.py` — implicit rewards + Bradley-Terry calibration
3. `03_Length_Bias_and_SimPO.py` — length-bias diagnostic, SimPO loss, VRAM benchmark
4. `04_Activation_Steering_and_Audit.py` — steering vectors + alignment tax audit

Each file is loaded as a module via `importlib`, so all logic lives in the `.py` deliverables and this notebook just orchestrates + reports.

## Setup

Adds the current directory to `sys.path` and imports shared helpers. Requires `torch`, `numpy`, `matplotlib`. Part 4 additionally needs `transformers` + `accelerate`.

In [ ]:
import sys, pathlib, importlib.util

WORKDIR = pathlib.Path.cwd()
sys.path.insert(0, str(WORKDIR))

def load(name, filename):
    spec = importlib.util.spec_from_file_location(name, WORKDIR / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

print('workdir:', WORKDIR)
print('python :', sys.version.split()[0])

## Part 1 — DPO Loss Core

Verifies `get_batch_logps` and confirms backprop only touches policy weights, leaving reference weights frozen.

In [ ]:
core = load('core', '01_DPO_Loss_Core.py')
core.test_get_batch_logps()
core.test_dpo_gradient_isolation()

## Part 2 — Implicit Reward & Bradley-Terry Evaluation

Computes implicit rewards $\hat r(x,y)=\beta\log\frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$, pairwise accuracy, and a reliability diagram.

In [ ]:
ev = load('ev', '02_Implicit_Reward_Evaluator.py')

batch = ev._synthetic_batch()
res = ev.evaluate_implicit_rewards(None, None, batch)
print('pairwise accuracy:', f"{res['pairwise_accuracy']:.4f}")

human = (res['bt_probabilities'] > 0.5).float().numpy()
curve = ev.bradley_terry_calibration_curve(res['bt_probabilities'], human)
print('\ncalibration bins (pred, empirical, n):')
for p, e, n in curve:
    print(f'  {p:.2f}  {e:.2f}  ({n})')

## Part 3 — Length Bias & SimPO

Runs the length correlation diagnostic, SimPO smoke test, and the DPO-vs-SimPO VRAM benchmark.

In [ ]:
import torch, numpy as np

sp = load('sp', '03_Length_Bias_and_SimPO.py')

# 3.1 Length correlation demo
np.random.seed(0)
len_w = np.random.randint(50, 200, size=256)
len_l = np.random.randint(50, 200, size=256)
margin = 0.4 * (len_w - len_l) + np.random.randn(256) * 5
r = sp.length_correlation(len_w, len_l, margin)
print(f'length-correlation r = {r:.3f} (positive => verbose bias)')

# 3.2 SimPO
torch.manual_seed(0)
p_w = -torch.rand(64) * 2 - 3
p_l = -torch.rand(64) * 2 - 3
loss, m = sp.SimP0Loss(beta=2.0, gamma=0.5)(p_w, p_l)
print(f'SimPO loss={loss.item():.4f}  mean margin={m.mean().item():.4f}')

# 3.3 VRAM benchmark
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sp.vram_benchmark(device=device)

## Part 4 — Activation Steering & Alignment Tax Audit

Loads a small instruct model, extracts a contrastive steering vector for the honest-vs-sycophantic direction at layer `L`, then sweeps $\alpha \in \{-5, -2.5, 0, +2.5, +5\}$ and measures truthfulness vs reasoning retention.  
Override the model with `os.environ['PS04_MODEL'] = '...'` before running this cell. First run downloads ~1 GB.

In [ ]:
import os
os.environ.setdefault('PS04_MODEL', 'Qwen/Qwen2.5-0.5B-Instruct')

st = load('st', '04_Activation_Steering_and_Audit.py')
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'loading {os.environ["PS04_MODEL"]} on {device} ...')
tokenizer = AutoTokenizer.from_pretrained(os.environ['PS04_MODEL'])
model = AutoModelForCausalLM.from_pretrained(
    os.environ['PS04_MODEL'], dtype=torch.float32
).to(device).eval()

layer_idx = min(8, len(model.model.layers) - 1)
v = st.extract_steering_vector(model, tokenizer, st._calibration_pairs(),
                              layer_idx=layer_idx, device=device)
print(f'steering vector shape={tuple(v.shape)}  layer={layer_idx}')

In [ ]:
rows = st.alignment_tax_audit(
    model, tokenizer, v, layer_idx,
    st._truth_eval_prompts(),
    alphas=(-5.0, -2.5, 0.0, 2.5, 5.0),
    device=device,
)

print('\nsummary:')
print(f"{'alpha':>6} {'truth%':>8} {'reason%':>8}")
for r in rows:
    print(f"{r['alpha']:>6.1f} {r['truthfulness']*100:>7.1f}% {r['reasoning']*100:>7.1f}%")

### Interpreting the audit

- Positive $\alpha$ pushes residual streams along the honest − sycophantic direction; truthfulness generally rises, but reasoning accuracy may fall if $\alpha$ is too large (the *alignment tax*).
- Record the range of $\alpha$ where truthfulness is maximised without meaningful reasoning loss — that is the recommended operating band for Subsystem 04.